# 02 - Calibration of the Player Generation Model

**Objective**: This notebook aims to extract some statistical laws underlying the carreers of tennis players from ATP results (1991-2024). We will extract parameters from our model to configure the players in our simulation later. 

**Method 1**: Each player will be attributed a score _S_ determined by an _intrinsic potential P_, calibrated on the distribution of maximum strengths of the players (based on the dataset available), and _aging A_, that is a function describing the evolution of strength with age.

## 0. Creation of Players Database Info

We create a dataset containing the main information about players' careers. We extract the maximum strength achieved by each player (from the `zermelo_strengths_1991-2024.csv` file) and the age at which this maximum strength was reached. We also store the age at first and last match played (We use the year of birth from the `atp_players.csv` file to compute ages). 

Since age is a crucial factor in our model, any player with missing birthdate information is excluded from the dataset. (The impact is minimal, as these kind of players have not played many matches in the dataset and have small Zermelo strengths).

For the players active in the boundaries of the dataset (1991 and 2024), we retain them in the dataset because they provide valuable information about the distribution of strengths. However, they will be excluded from the calibration of aging curves and carreer duration later, since we do not have their full career data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

In [ ]:
zermelo_strengths_data = pd.read_csv("../data/processed/zermelo_strengths_1991-2024.csv")
players_info_data = pd.read_csv("../data/tennis_atp/atp_players.csv", low_memory=False)
#display(players_info_data.head())

In [ ]:
# sort the data so we can get the year corresponding to the maximum strength on the first line
zermelo_strengths_data_sorted = zermelo_strengths_data.sort_values("zermelo_strength", ascending=False)

# group the data by IDs, and then get: start and end years, max strength and corresponding year
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html
players_stats_data = zermelo_strengths_data_sorted.groupby("player_id").agg(start_year=("year", "min"), 
                                                       end_year=("year", "max"), 
                                                       top_year=("year", "first"),
                                                       top_strength=("zermelo_strength", "max")).reset_index()                     

#display(players_stats_data)

In [ ]:
# formatting year of birth to get only the year (with .dt.year)
# https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
# https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior
players_info_data["birth_year"] = pd.to_datetime(players_info_data["dob"].astype(str), format="%Y%m%d.0", errors="coerce").dt.year

In [ ]:
players_full_data = pd.merge(players_stats_data, players_info_data[["player_id","birth_year", "name_first", "name_last"]], on="player_id", how="left")
players_full_data = players_full_data.dropna(subset = ["birth_year"]) # delete all the players without a valid birth year 
players_full_data["birth_year"] = players_full_data["birth_year"].astype(int) # to remove the .0 after the year


# add start_age / top_age / end_age
players_full_data["start_age"]= players_full_data["start_year"]-players_full_data["birth_year"]
players_full_data["end_age"]= players_full_data["end_year"]-players_full_data["birth_year"]
players_full_data["top_age"]= players_full_data["top_year"]-players_full_data["birth_year"]


# removal of players who have a negative age or are younger than 14 years old
# It seems that different players have the same ID, so they are deleted (e.g. son confused with his father)
# example: Martin Damm (father born in 1972, son born in 2003 --> year displayed: 2003)
players_full_data = players_full_data[players_full_data["start_age"]>=14]

#display(players_full_data.head())

In [ ]:
#check whether a player was active both in 1991 and in 2024
all_years_player = players_full_data[(players_full_data["start_year"]==1991) & (players_full_data["end_year"]==2024)]
print("Number of players active both in 1991 and in 2024:", len(all_years_player))

# Detecting complete careers: new column added in the data to find the players whose start and end are available 
# (i.e. who started after 1991 and ended before 2024)

players_full_data["career_type"] = np.where((players_full_data["start_year"]>1991) & 
                                                (players_full_data["end_year"]<2024), 
                                                "full", "Other")

players_full_data["career_type"] = np.where(players_full_data["start_year"]==1991, "left_boundary",
                                               players_full_data["career_type"])

players_full_data["career_type"] = np.where(players_full_data["end_year"]==2024, "right_boundary",
                                               players_full_data["career_type"])


full_career_players_nbr = (players_full_data["career_type"]=="full").sum() 
valid_careers_nbr = len(players_full_data)

print("Number of complete careers:", full_career_players_nbr, "out of", valid_careers_nbr, 
      f"({100*full_career_players_nbr/valid_careers_nbr:.2f}%)")

In [ ]:
# rearranging the order of the columns
players_full_data = players_full_data[["player_id", "name_first", "name_last", 
                                       "birth_year", "start_year", "end_year", "top_year", 
                                       "start_age", "end_age", "top_age", 
                                       "top_strength", "career_type"]]

In [ ]:
# saving the file in a .csv
output_path = "../data/processed/players_stats.csv"
players_full_data.to_csv(output_path, index=False)

All the information is stored in a new dataframe `players_full_data`. It contains the following columns: $\\$
$\textbf{player\_id, name\_first, name\_last, birth\_year, start\_year, end\_year, top\_year, start\_age, end\_age, top\_age, top\_strength, complete\_career}$.

## 1. Testing the Stationarity of the Player Generation Process

Before calibrating our model, we must verify that the process generating players is stationary over time. This verification stands on the hypothesis that the distribution of players' intrinsic potentials (maximum strengths) remains consistent across different time periods, and that the number of new players entering the professional circuit each year is relatively stable. 

### 1.1. Stationarity of Maximum Strengths Distribution (Quality)

We first check that the distribution of players' strengths is stationary over time. We plot the distribution of players' maximum strengths for different time periods and compare them. To validate this hypothesis, we examine the evolution of `top_strength` based on the player's `start_year` by focusing on the average level stability (median) and the elite level stability (top 10% players). The overall shape of distribution over decades is also analysed to ensure no significant shifts occur.

If this hypothesis holds, we can pool all players together to estimate the distribution of intrinsic potentials.

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


# plot of the median trend line
sns.lineplot(data=players_full_data,
             x="start_year", y="top_strength", 
             estimator="median", errorbar=("pi",50), # colored band containing 50% of the points around the median
             color="#e74c3c",linewidth=3,
             label="Median Strength",
             zorder=3)

# plot of the top 10% trend line
sns.lineplot(data=players_full_data, 
             x="start_year", y="top_strength", 
             estimator=lambda x: np.percentile(x, 90), # 90th percentile = top 10% (value )
             errorbar=("ci", 95), # confidence interval of 95%
             label="90th Percentile Strength",
             color="#8e44ad", linewidth=3, linestyle="--", zorder=4)

# plot of the maximum trend line
sns.lineplot(data=players_full_data, x="start_year", y="top_strength", 
             estimator=np.max, errorbar=None, 
             label="Max Strength",
             color="#f1c40f", linewidth=3, linestyle="-.", zorder=5)


# plot of the individual points
sns.stripplot(data=players_full_data, 
              x="start_year", y="top_strength", 
              size=2.5, 
              hue="career_type",
              hue_order=["full", "left_boundary", "right_boundary"],
              palette=["#687475", "#60a215e8", "#1484cf"], 
              jitter=0.3, #to avoid overplotting
              native_scale=True, 
              zorder=2)

plt.title("Distribution of Player Maximum Zermelo Strengths by Career Start Year", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12)

plt.yscale("log")
plt.ylabel(r"Maximum Zermelo Strength ($\pi_{max}$)", fontsize=18)

# legend (keep only the relevant legend items)
handles, _ = plt.gca().get_legend_handles_labels()

handles_curves = [handles[0], handles[1], handles[2]]
labels_curves = ["Median (IQR band)", "Top 10% (95% CI)", "Top 1"]
legend_curves = plt.legend(handles=handles_curves, 
                           labels=labels_curves, 
                           markerscale=2.5, fontsize=13,
                           loc="upper right",
                           frameon=False)

plt.gca().add_artist(legend_curves) # to keep both legends


handles_points = [handles[4], handles[3], handles[5]]
labels_points = ["Left bounded (start in 1991)", "Complete Career", "Right bounded (end in 2024)"]
plt.legend(handles=handles_points, 
                           labels=labels_points, 
                           markerscale=3, fontsize=13,
                           bbox_to_anchor=(0.8, -0.15),
                           frameon=True, ncol=3)


plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

<font color="blue"> **_The figure strongly supports our stationary hypothesis!_** </font> The median player strength (in red) and the elite threshold (in purple) remain horizontal and stable from 1992 to 2018, proving that the generation of talent has not changed structurally over time. 

The high variance in 1991 comes from players already in their prime (they started before 1991, but the first year of our dataset is 1991). We observe a slight increase in 2019, immediately followed by a decrease in 2020 (likely due to consequences of the COVID-19 pandemic). After this year, it continues decreasing due to right-censoring: players starting their carreers recently have not yet reached their real maximum strength. 

Regarding the elites, the top 10% threshold mirrors the stabilitiy of the median, except with some small fluctuations. It confirms that the top players can be generated consistently year over year. 

The Top 1 curve (in yellow) is more volatile, with peaks rather than a specific trend. This is expected, as the very best players can vary significantly from year to year due to the emergence of exceptional talents or the dominance of a few players.

---

To ensure robutstness, we must restrict the calibration of our model parameters to a stable period **1992-20YY**, where the stationary hypothesis is most valid. To determine the cut-off year, we estimate the time required for players to reach their potential, by taking players that started between 1992 and 2002.  It will be done by calcutating the years to reach their maximum strength. 

In [ ]:
# finding the players that begin between 1992 and 2002
years_to_top_players = players_full_data[(players_full_data["start_year"]>=1992) & (players_full_data["start_year"]<=2002)].copy()

# calculating the years to reach top
years_to_top_players["years_to_top"] = years_to_top_players["top_year"]-years_to_top_players["start_year"]

years_to_top_players["years_to_top"].describe(percentiles=[0.25,0.5,0.9, 0.95, 0.99])

The median tie to reach peak strength is only 1 year, and the first quartile is 0 year. This indicates that the majority of players do not experience a long development: they enter the tour, reach their maximum almost immediately and decline.

The maximum value of 31 years seems to be unrealistic. The 95th percentile lies at 10 years. This means that only 5% of players take more than 10 years to reach their peak, which is a reasonable threshold to consider for our cut-off year.

<font color="purple"> Calibration is made on the starting year period **1992-2014**! </font> 
$\newline$ _to ensure that all players (the vast majority) have reached their potential._

In [ ]:
calibration_start = 1992
calibration_end = 2014

Now, we want to see if the distribution of maximum strengths is consistent across generations. We plot the distribution of `top_strength` for players starting in different decades (1990s, 2000s, 2010s).

In [ ]:
calibration_players = players_full_data[(players_full_data["start_year"]>=calibration_start) & 
                                        (players_full_data["start_year"]<=calibration_end)].copy()

In [ ]:
# getting the decades of the players (1990s, 2000s, 2010s)
calibration_players["generation"] = calibration_players["start_year"] - (calibration_players["start_year"] % 10)
#display(calibration_players["generation"].value_counts())

# creation of the labels for the generations
generations_min = calibration_players.groupby("generation")["start_year"].min()
generations_max = calibration_players.groupby("generation")["start_year"].max()

labels = []

for generation in generations_min.index:
    start = generations_min[generation]
    end = generations_max[generation]
    label = f"{generation}s ({start}-{end})" if start != end else f"{generation}s ({start})"
    labels.append(label)

calibration_players["generation_label"] = calibration_players["generation"].replace(generations_min.index, labels)

calibration_players = calibration_players.sort_values("generation")
hue_order = calibration_players["generation_label"].unique()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plot of each distribution by generation
sns.kdeplot(data=calibration_players, 
            x="top_strength", 
            hue="generation_label", 
            hue_order=hue_order,
            log_scale=True, fill=False,
            common_norm=False,
            linewidth=4, alpha=1,
            palette="viridis",
            zorder=2)

# plot of the global distribution (all players) if show_global is True
show_global = False

if show_global:
    sns.kdeplot(data=calibration_players, 
                x="top_strength",
                log_scale=True, fill=False,
                common_norm=False,
                linewidth=2.5, alpha=0.9,
                color="red",
                linestyle="--",
                zorder=3)

plt.title("Maximum Zermelo Strength Distribution by Generation", fontsize=25, weight="bold", pad=35)

plt.xlabel(r"Maximum Zermelo Strength ($\pi_{max}$)", fontsize=18)
plt.xticks(fontsize=12)

plt.ylabel("Density", fontsize=18)

# legend
handles = plt.gca().get_lines()
all_labels = list(hue_order)

if show_global:
    all_labels.append("Global Distribution")


legend = plt.legend(handles=handles, title="Career Start Decade", 
                    labels=all_labels, 
                    fontsize=13, title_fontsize=14, 
                    loc="upper center", frameon=False)
plt.setp(legend.get_title(), fontweight='bold')


plt.grid(visible=True, which="major", axis="x", color="gray", linewidth=0.5, alpha=0.5)
plt.grid(visible=True, which="minor", axis="x", color="gray", linestyle=':', linewidth=0.5, alpha=0.3)

sns.despine()
plt.tight_layout()
plt.show()

The KDE plot reveals similarity across all three decades. The distributions share a similar shape, with peaks around the same strengths, followed by a heavy right tail. While the blue curve (2010s) appears a bit shifted, the overall overlap  between the distributions suggest that the process of player generation has remained consistent across these decades.

In [ ]:
stats = calibration_players.groupby("generation_label")["top_strength"].describe()

stats[["count", "mean", "50%", "std", "max"]]

The `mean` and `50%` (median) value is quite stable across decades, with a slight decrease in the 2010s. 

By looking at the `std` column, it suggests that the system is chaotic. It drops significantly from the 2000s to the 2010s (by a factor of 3)! It would mean that the distributions changed a lot, but here it simply means that linear statistics are not sufficient to capture the distribution of strengths. There is a correlation between the maximum strength and the standard deviation!

The standard deviation is dominated by the right tail of the distribution, and is influenced a lot by the presence of a few very strong players. We should move to the logarithmic scale:

In [ ]:
calibration_players["log10_strength"] = np.log10(calibration_players["top_strength"])

calibration_players.groupby("generation_label")["log10_strength"].std()

When analysed in the log scale, the standard deviation is much more stable across decades, confirming that the apparent increase in variability was due to the presence of a few outliers in the right tail of the distribution. 

The log transformation helps to stabilize the variance and helps for finding theunderlying distribution of player strengths and confirming that the generation process has remained consistent over time.

<font color="purple"> _Conclusion_: **We can therefore validate the stationary hypothesis of maximum strength distribution**, and pool all players together to estimate the distribution of intrinsic potentials. We will use the log10 of `top_strength` for the calibration of our model parameters. </font>

In [ ]:
# saving the calibration players data in a .csv

output_calibration_path = f"../data/processed/calibration_players_{calibration_start}-{calibration_end}.csv"
calibration_players.to_csv(output_calibration_path, index=False)

### 1.2. Stationarity of the Number of New Players (Quantity)

To ensure having a realistic simulation, we need to model the arrival of new players. Instead of a fixed number, we model the incoming flux using a Gaussian distribution $\mathcal{N}(\mu, \sigma)$, where $\mu$ is the average number of new players per year and $\sigma$ is the standard deviation of the number of new players per year. (It will be shown that this distribution is a good fit for the data).

In [ ]:
# counting the number of players starting each year in the calibration period
numbers_per_year = calibration_players.groupby("start_year")["start_year"].count()

# calculating the mean and standard deviation of the number of new players per year
arrival_mean = numbers_per_year.mean()
arrival_std = numbers_per_year.std()

print(f"Average number of new players per year (\u03bc): {arrival_mean:.1f}")
print(f"Standard deviation of new players per year (\u03c3): {arrival_std:.1f}")

Here, we see that $\sigma$ >> $\sqrt{\mu}$ $(\approx 26.1$), which means that a Poisson distribution would not be appropriate there. To see if the Gaussian distribution is a good fit, we can do a Shapiro-Wilk test (more robust than the Kolmogorov-Smirnov test for small samples) for normality. $\textbf{The null hypothesis is that the data is normally distributed.}$ If the p-value is greater than a significance level (e.g., 0.05), we fail to reject the null hypothesis, suggesting that the data is consistent with a normal distribution. But this test is not sufficient, so we will also look at the histogram and the Q-Q plot of the data to visually assess the normality.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html
from scipy.stats import shapiro

shapiro_stat, shapiro_p_value = shapiro(numbers_per_year)

print(f"Shapiro-Wilk Test Statistic: {shapiro_stat:.4f}")
print(f"Shapiro-Wilk Test p-value: {shapiro_p_value:.4f}")

if shapiro_p_value > 0.05:
    print("\nFail to reject the null hypothesis: the data is consistent with a normal distribution.")
else:
    print("\n Reject the null hypothesis: the data is not consistent with a normal distribution.")

In [ ]:
# histogram of the number of new players per year

plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

sns.barplot(x=numbers_per_year.index, y=numbers_per_year.values, color="#3498db", 
            alpha=0.6, zorder=2, edgecolor="black")

# horizontal line for the mean value
plt.axhline(y=arrival_mean, 
            color="#e74c3c", label=rf"Mean value: $\mu$={arrival_mean:.1f}", 
            zorder=3, linewidth=3, linestyle="-")

# zone containing the mean value +/- 2 standard deviations (95% of the data if normal distribution)
plt.axhspan(ymin=arrival_mean - 2*arrival_std, 
            ymax=arrival_mean + 2*arrival_std, 
            color="#e74c3c", alpha=0.1, label=rf"95% CI ($\mu \pm 2\sigma$, $\sigma = {arrival_std:.1f}$)", zorder=0)

plt.title(f"Flux of New Players Entering the Tour per Year ({calibration_start}-{calibration_end})", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12, ticks=range(0, len(numbers_per_year),3))

plt.ylabel("Number of New Players ($N_{new}$)", fontsize=18)


plt.grid(visible=True, axis="y", linewidth=0.5, alpha=0.7, zorder=0)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

We observe that the shaded area representing the 95% confidence interval (mean ± 2 standard deviations) covers most of the data points, which is consistent with the properties of a normal distribution.

To confirm visually the result of the test (as the number of data points is quite small), we use a Q-Q Plot. It takes our data, sort it in ascending order, and plots it against the quantiles of a normal distribution, that is matched to the size of the data. If the points in the Q-Q plot approximately lie on a straight line, it suggests that the data is normally distributed.

To confirm visually the result of the test (as the number of data points is quite small), we use a Q-Q Plot:
 
- X axis: Theoretical quantiles from  a standard normal distribution $\mathcal{N}(0,1)$, corresponding to a division of the data into $n$ slices of equal probability (where $n$ is the number of data points). For each slice $i$, we calculate $P_i$=$(i-0.5)/n$, which gives us the cumulative probability up to that slice. We then convert it into a Z-score using the probit function (the inverse of the cumulative distribution function): $Z_i$ = $\Phi^{-1}(P_i)$. It representes the theoretical distance from the mean measured in units of standard deviation.


- Y axis: The actual data points, sorted in ascending order. Each point corresponds to the number of new players in a given year, ordered from the smallest to the largest.

- Red line: The line represents the expected relationship if the data were perfectly normally distributed, i.e. Y= $\mu + \sigma X$. If the points closely follow this line, it suggests that the data is consistent with a normal distribution.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html
from scipy.stats import probplot

# osm and osr: tuple of theoretical quantiles and ordered values of the data, used for plotting the Q-Q plot
# slope and intercept and r: standard deviation, mean and correlation coefficient of the data (used for the red line in the Q-Q plot)
(osm, osr), (slope, intercept, r) = probplot(numbers_per_year, dist="norm")

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plot of points in the Q-Q plot
plt.scatter(osm, osr, color="#3498db", edgecolor="black", s=200, zorder=3)

# plot of the red line representing the expected relationship if the data were perfectly normally distributed
x_line = np.array([min(osm), max(osm)])
y_line = intercept + slope * x_line
plt.plot(x_line, y_line, color="#e74c3c", linewidth=3, zorder=2,
         label=r"Normal Reference Line ($Y = \mu + \sigma X$)")



plt.title("Normal Q-Q Plot of the Number of New Players per Year", fontsize=25, weight="bold", pad=35)
plt.xlabel("Theoretical Quantiles X (Standard Z-scores)", fontsize=18)
plt.ylabel("Data Quantiles Y (Number of New Players)", fontsize=18)


plt.grid(visible=True, axis="y", linewidth=0.5, alpha=0.7, zorder=0)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper left")
plt.tight_layout()
plt.show()

This graph shows a strong linear trend. Most of the data points are closely aligned with the red reference line, particularly in the $[-1.5,1.5]$ range. This indicates that the core of the distribution matches the Normal model very well. We observe slight deviations at the extremities, especially for the maximum value. It suggest that these extreme values occur more frequently in reality than a perfrect normal distribution would predict.

Finally, we can calculate the coefficient of determination $R^2$, which measures the goodness of fit of our data to the normal data. Here, it is the correlation between the theoretical normal quantiles and the observed quantiles. A value close to 1 indicates that the normal distribution is a highly accurate representation of the data.  

In [ ]:
print(f"R^2 = {r**2:.3f}")

The coefficient is above $0.9$, which is a strong validation of our distribution choice! 

<font color="purple">  Based on this analysis, we can conclude that the incoming flux in our model each year is modelled by the normal distribution $$\mathcal{N}(\mu_{arrival}, \sigma_{arrival}).$$ </font>

In the simulation, the values drawn from this distribution will be rounded to the nearest integer. 

In [ ]:
# saving the arrival parameters (mean and std) in .json file

arrival_data = {
    "arrival_params": {
    "mu" : arrival_mean,
    "sigma" : arrival_std, 
    "calibration period" : [calibration_start, calibration_end]
                            }
                }

config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        params = json.load(f)
else:
    params = {}

params.update(arrival_data)

os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(params, f, indent=3)

## 2. Calibrating the Distribution of Intrinsic Potentials (Talent)

## 3. Calibrating the Aging Curves (Evolution of Strength with Age)

## 4. Calibrating Career Duration and Retirement

## Sources

$\textbf{1.2. Stationarity of the Number of New Players (Quantity)}$

- Ghasemi A., & Zahediasl S. (2012). $\textit{Normality Tests for Statistical Analysis: A Guide for Non-Statisticians}$, International Journal of Endocrinology and Metabolism, 10(2), 486–489.
https://pmc.ncbi.nlm.nih.gov/articles/PMC3693611/ $\newline$
(for justifying the combination of graphic inspection + statistical tests for normality, because if statistical tests with a p-value only can lead to erroneous conclusions)
- Razali, N. M., & Wah, Y. B. (2011). $\textit{Power comparisons of Shapiro-Wilk, Kolmogorov-Smirnov, Lilliefors and Anderson-Darling tests.}$ Journal of Statistical Modeling and Analytics, 2(1), 21–33. https://www.nrc.gov/docs/ml1714/ml17143a100.pdf $\newline$ (for justifying the choice of Shapiro-Wilk test for normality, small samples)
- Shapiro, S. S., & Wilk, M. B. (1965). $\textit{An Analysis of Variance Test for Normality (Complete Samples).} Biometrika, 52(3/4), 591–611. https://doi.org/10.2307/2333709 $\\newline$
(for the original paper introducing the Shapiro-Wilk test)
- Wilk, M. B., & Gnanadesikan, R. (1968). Probability Plotting Methods for the Analysis of Data. Biometrika, 55(1), 1–17. https://doi.org/10.2307/2334448 $\\newline$
(for the original paper introducing the Q-Q plot)


